In [3]:
import os
import shutil
from pathlib import Path
from PIL import Image

# --- CONFIG ---
tasks = [
    (Path("data-files/Storke"), Path("data-webp/Storke")),
    (Path("data-files/LEDs"), Path("data-webp/LEDs")),
]
quality = 80
max_width = 1920  # resize any image wider than this

# --- Helper to test WebP support ---
def check_webp_support():
    try:
        tmp = Path("test.webp")
        Image.new("RGB", (1, 1), color="white").save(tmp, "webp")
        tmp.unlink(missing_ok=True)
        return True
    except Exception as e:
        print("⚠️ WebP test failed:", e)
        return False

if not check_webp_support():
    raise RuntimeError("Your Pillow build lacks WebP support. Try reinstalling: pip install --upgrade pillow")

# --- Conversion ---
total_converted, total_moved, total_failed = 0, 0, 0

for source_dir, target_dir in tasks:
    target_dir.mkdir(parents=True, exist_ok=True)
    count, moved, failed = 0, 0, 0

    for root, _, files in os.walk(source_dir):
        for file in files:
            file_lower = file.lower()
            src_path = Path(root) / file
            rel_path = src_path.relative_to(source_dir)
            if file_lower.endswith(".webp"):
                dst_path = target_dir / rel_path
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                try:
                    shutil.move(str(src_path), str(dst_path))
                    moved += 1
                    print(f"✅ Moved {src_path} → {dst_path}")
                except Exception as e:
                    failed += 1
                    print(f"❌ Failed to move {src_path}: {e}")
                continue
            if not file_lower.endswith((".jpg", ".jpeg", ".png")):
                continue
            dst_path = target_dir / rel_path.with_suffix(".webp")
            dst_path.parent.mkdir(parents=True, exist_ok=True)

            try:
                with Image.open(src_path) as img:
                    img = img.convert("RGB")

                    # Optional resize for huge images
                    if img.width > max_width:
                        ratio = max_width / img.width
                        new_size = (max_width, int(img.height * ratio))
                        img = img.resize(new_size, Image.LANCZOS)

                    img.save(dst_path, "webp", quality=quality, method=6)
                count += 1
                print(f"✅ {src_path} → {dst_path}")
            except Exception as e:
                failed += 1
                print(f"❌ Failed on {src_path}: {e}")

    print(f"\n✅ Done for {source_dir}! {count} images converted, {moved} webp moved, {failed} failed.")
    total_converted += count
    total_moved += moved
    total_failed += failed

print(f"\n✅ Total: {total_converted} images converted, {total_moved} webp moved, {total_failed} failed.")

✅ data-files\Storke\Storke 1.png → data-webp\Storke\Storke 1.webp

✅ Done for data-files\Storke! 1 images converted, 0 webp moved, 0 failed.
✅ data-files\LEDs\Costume Over.jpeg → data-webp\LEDs\Costume Over.webp
✅ data-files\LEDs\Costume Underneath.jpeg → data-webp\LEDs\Costume Underneath.webp
✅ data-files\LEDs\Diagram.png → data-webp\LEDs\Diagram.webp
✅ data-files\LEDs\LED Image 1.JPG → data-webp\LEDs\LED Image 1.webp
✅ data-files\LEDs\LED Image 2.JPG → data-webp\LEDs\LED Image 2.webp
✅ data-files\LEDs\LED Image 3.JPG → data-webp\LEDs\LED Image 3.webp
✅ data-files\LEDs\LED PATTERNS.png → data-webp\LEDs\LED PATTERNS.webp
✅ data-files\LEDs\Pixelblaze UI.png → data-webp\LEDs\Pixelblaze UI.webp

✅ Done for data-files\LEDs! 8 images converted, 0 webp moved, 0 failed.

✅ Total: 9 images converted, 0 webp moved, 0 failed.
